In [43]:
import numpy as np
import pandas as pd
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

In [44]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\sampled_data\sampled_data.csv")

In [45]:
df.sample(3)

,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),FastCharge_time_hrs,Energy_weight_ratio,Added_Range_1Stop,Range_km_level,Price_level
1175,0.516129,0.782274,0.102941,0.595331,0.695853,0.575758,0.666667,0.1325,0.201072,0.259046,0.613679,0.633065,1,2
1156,0.489247,0.413295,0.519608,0.207523,0.281106,0.151515,0.500000,0.2755,0.158177,0.492188,0.340986,0.294355,0,0
700,0.435484,0.744701,0.107843,0.814527,0.777880,0.993939,0.750000,0.2715,0.218499,0.164896,0.736159,0.991935,2,2


# Correlation with target

In [46]:
import pandas as pd

numeric_features = df.select_dtypes(include=[np.number]).drop(columns=['Range_km_level'])
target = df['Range_km_level']

# Pearson
pearson_corr = df[numeric_features.columns].corrwith(target)
# print("Pearson Correlation with Range km level\n", pearson_corr.sort_values(ascending=False))

# Spearman 
spearman_corr = df[numeric_features.columns].corrwith(target, method='spearman')
# print("\nSpearman Correlation with range km level \n", spearman_corr.sort_values(ascending=False))

#--------------------------------------------------------------------------------------------------------

numeric_features1 = df.select_dtypes(include=[np.number]).drop(columns=['Price_level'])
target1 = df['Price_level']

# Pearson
pearson_corr1 = df[numeric_features1.columns].corrwith(target1)
# print("Pearson Correlation with Range km level\n", pearson_corr1.sort_values(ascending=False))

# Spearman 
spearman_corr1 = df[numeric_features1.columns].corrwith(target1, method='spearman')
# print("\nSpearman Correlation with range km level \n", spearman_corr1.sort_values(ascending=False))

# ---------------------------------------------------------------------------------------------------

data = {
    "Range_level_Pearson": pearson_corr,
    "Range_level_Spearman" : spearman_corr,
    "Price_level_Pearson": pearson_corr1,
    "Price_level_Spearman" : spearman_corr1
}

table = pd.DataFrame(data).sort_values(by='Range_level_Pearson', ascending=False)

In [47]:
table

,Range_level_Pearson,Range_level_Spearman,Price_level_Pearson,Price_level_Spearman
Firth_stop_range_km,0.878883,0.915032,0.603611,0.616690
Battery_kWh,0.803766,0.828120,0.780328,0.801091
Energy_weight_ratio,0.787358,0.851513,0.556228,0.577400
Fast_charger(kW),0.683849,0.778251,0.612024,0.686962
Weight_kg,0.605164,0.607364,0.823137,0.858283
Price_level,0.600963,0.600283,NaN,NaN
Added_Range_1Stop,0.589277,0.616195,0.384328,0.368961
Towing_kg,0.369383,0.365680,0.399129,0.391455
Cargo_volume,0.159477,0.293549,0.269424,0.389614
Price/Range(km),0.006909,-0.014153,0.512313,0.597246


# Correlation with features

In [48]:
import plotly.express as px
import numpy as np
import pandas as pd

corr_matrix = df[numeric_features.columns].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,          # Show correlation numbers
    color_continuous_scale='RdBu_r',
    origin='upper',
    labels=dict(x="Features", y="Features", color="Correlation"),
    title="Feature-to-Feature Correlation (Numeric Features)"
)
fig.update_xaxes(side="top")
fig.show()

upper = corr_matrix.abs().where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
print("Highly correlated features to drop:", to_drop)


Highly correlated features to drop: ['Battery_kWh', 'Energy_weight_ratio', 'Added_Range_1Stop']


In [49]:
df_new = df.drop(columns=['Added_Range_1Stop'])

# Feature importance with model(Random forrest)

In [50]:
x = df_new.drop(columns=['Range_km_level','Price_level'])
y = df[['Range_km_level','Price_level']]

In [51]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
xb = MultiOutputClassifier(XGBClassifier(random_state=99))
xb.fit(x,y)

Range_feat_importance = pd.Series(xb.estimators_[0].feature_importances_, index=x.columns).sort_values(ascending=False)
Price_feat_importance = pd.Series(xb.estimators_[1].feature_importances_,index=x.columns).sort_values(ascending=False)

print("XGboost Range Feature Importance:\n", Range_feat_importance) 
print("XGBoost Price feature importance:\n ", Price_feat_importance)

XGboost Range Feature Importance:
 Firth_stop_range_km    0.556927
Energy_weight_ratio    0.107028
FastCharge_time_hrs    0.090446
Fast_charger(kW)       0.081228
Battery_kWh            0.063240
Weight_kg              0.025484
Efficiency             0.021060
Towing_kg              0.020903
Acceleration(0-100)    0.014410
Cargo_volume           0.011053
Price/Range(km)        0.008220
dtype: float32
XGBoost Price feature importance:
  Weight_kg              0.527117
Towing_kg              0.132332
Price/Range(km)        0.064835
Battery_kWh            0.056879
Firth_stop_range_km    0.055179
Acceleration(0-100)    0.036934
Energy_weight_ratio    0.036287
Fast_charger(kW)       0.029797
Efficiency             0.025331
Cargo_volume           0.018617
FastCharge_time_hrs    0.016691
dtype: float32
